In [31]:
from pyspark.sql import functions as F

df_bronze_teams = spark.sql("select * from bronze_teams")
df_silver_teams = df_bronze_teams.select("id", "name", "shortName", "tla", "crest")

df_silver_players = (df_bronze_teams
    # Explode diretamente a coluna 'squad' que já é um array nativo
    .select("id", F.explode("squad").alias("player"))
    
    .select(
        F.col("id").alias("id_team"), 
        F.col("player.id").alias("id_player"),
        F.col("player.name").alias("name"),
        F.col("player.dateOfBirth").alias("dateOfBirth"),
        F.col("player.nationality").alias("nationality"),
        F.col("player.position").alias("position")
    )
)

StatementMeta(, 98d501d1-e6aa-4b5f-b9b2-c54a7f725251, 33, Finished, Available, Finished, False)

In [33]:
display(df_bronze_teams)

StatementMeta(, 98d501d1-e6aa-4b5f-b9b2-c54a7f725251, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38b35458-5974-4e0c-9363-92170b825290)

In [23]:
df_silver_teams.write.mode('overwrite').saveAsTable("silver_teams")
df_silver_players.write.mode('overwrite').saveAsTable("silver_players")

StatementMeta(, 98d501d1-e6aa-4b5f-b9b2-c54a7f725251, 25, Finished, Available, Finished, False)

In [24]:
from pyspark.sql import functions as F

df_bronze_matches = spark.sql("select * from bronze_matches")
df_silver_matches = (df_bronze_matches.where("stage = 'GROUP_STAGE'")
    #.select("id", "utcDate", "stage", "homeTeam.id", "awayTeam.id")
    .select(
        F.col("id").alias("id"),
        F.col("utcDate").alias("utcDate"),
        F.col("stage").alias("stage"),
        F.col("homeTeam.id").alias("homeTeam"),
        F.col("awayTeam.id").alias("awayTeam")
    )
)
df_silver_matches.write.mode('overwrite').saveAsTable("silver_matches")

StatementMeta(, 98d501d1-e6aa-4b5f-b9b2-c54a7f725251, 26, Finished, Available, Finished, False)

In [27]:
def getTeamMatches(teamName):
    # O f-string do Python injeta o parâmetro direto na query SQL
    query = f"""
        SELECT  
    m.*, th.name as homeTeam, ta.name as awayTeam
FROM silver_matches m
    left join silver_teams th on th.id = m.homeTeam
    left join silver_teams ta on ta.id = m.awayTeam
where 
    th.name = '{teamName}' or ta.name = '{teamName}' 
    """
    return spark.sql(query)

StatementMeta(, 98d501d1-e6aa-4b5f-b9b2-c54a7f725251, 29, Finished, Available, Finished, False)